# KinoCompass – векторы e5 для поиска на сайте

Шаг 6д: векторы multilingual-e5-small для всех 23 349 фильмов каталога по **всем** тегам набора (сайт знает всё до 12.10.2023) – среднее векторов английского и русского текста «название. жанры. теги. описание», как лучший вариант проверки поиска (шаг 5в). На GPU – минуты, на процессоре – около часа.

Запускает `python export.py e5`; в конце скачается архив `export_e5.zip` (`e5_enru.npy` – векторы float16, `e5_ids.npy` – id фильмов) – его нужно распаковать в `model/export/`.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все».

Время на T4 (оценка): загрузка набора – около 1 минуты, модель e5 – около минуты, векторы – 3–8 минут.

In [ ]:
# каталог – где хранить модель e5 и посчитанные векторы внутри Colab
CACHE = "/content/cache"

In [ ]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

In [ ]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1 sentence-transformers==6.2.0

In [ ]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research CACHE={CACHE} python -W ignore export.py e5

In [ ]:
from google.colab import files

!cd /content/KinoCompass/model/export && zip -q /content/export_e5.zip e5_enru.npy e5_ids.npy && ls -la e5_enru.npy e5_ids.npy
files.download("/content/export_e5.zip")